In [ ]:
!pip install pyngrok streamlit

from pyngrok import ngrok
import subprocess
import time

# Kill any existing tunnels
ngrok.kill()
ngrok.set_auth_token("3B79BU5EVVSseYMZebZhX4Ms7zc_5mi9E3D4syHwJFKJ7QL9P")

# Start Streamlit in background
proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

# Create tunnel with static domain
public_url = ngrok.connect(8501, domain="unpostered-amiee-pangenetically.ngrok-free.dev")
print(f"\n✅ App is live! Click this link to open:\n")
print(f"👉 https://unpostered-amiee-pangenetically.ngrok-free.dev\n")

# Student Success Copilot

**A hybrid AI system using Search, Rule-Based Reasoning, and Machine Learning**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tordev1/Ai_repo/blob/main/Student_Success_Copilot.ipynb)

---

This notebook implements a **Student Success Copilot** that combines multiple AI techniques to assess student risk and provide actionable recommendations:

| Component | Technique | Purpose |
|---|---|---|
| Data Generator | Statistical Simulation | Generate 1200 synthetic student records |
| ML Model | Random Forest Classifier | Predict risk level from student features |
| Rule Engine | Forward & Backward Chaining (22 rules) | Expert-system reasoning with explanations |
| Search Planner | A* and Greedy Best-First Search | Optimise weekly study schedules |
| Fuzzy Logic | Fuzzy Inference System | Nuanced risk assessment with soft boundaries |

## 0. Install Dependencies

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn scikit-fuzzy networkx joblib

In [ ]:
import warnings
warnings.filterwarnings('ignore')

---
## 1. Data Generation

We generate **1200 synthetic student records** with realistic distributions for stress, confidence, workload, study hours, and deadline proximity. Risk labels are derived using a scoring formula with controlled noise.

In [ ]:
import pandas as pd
import numpy as np

SEED = 42
NUM_RECORDS = 1200


def generate_dataset(n=NUM_RECORDS, seed=SEED):
    rng = np.random.RandomState(seed)

    genders = rng.choice(["male", "female", "non-binary"], size=n, p=[0.45, 0.45, 0.10])

    # Workload: number of active modules/subjects (1-6)
    workload = rng.randint(1, 7, size=n)

    # Study hours per week (2-40, skewed toward lower values)
    study_hours = np.clip(rng.gamma(4, 3, size=n), 2, 40).round(1)

    # Confidence (1-10 scale)
    confidence = np.clip(rng.normal(6, 2, size=n), 1, 10).round(1)

    # Stress (1-10 scale)
    stress = np.clip(rng.normal(5.5, 2.5, size=n), 1, 10).round(1)

    # Days to nearest deadline (1-60)
    days_to_deadline = rng.randint(1, 61, size=n)

    # --- Derive risk label from features using logical rules ---
    risk = []
    for i in range(n):
        score = 0.0

        # High stress increases risk
        if stress[i] >= 8:
            score += 3
        elif stress[i] >= 6:
            score += 1.5

        # Low confidence increases risk
        if confidence[i] <= 3:
            score += 3
        elif confidence[i] <= 5:
            score += 1.5

        # Low study hours relative to workload
        ratio = study_hours[i] / max(workload[i], 1)
        if ratio < 3:
            score += 2.5
        elif ratio < 5:
            score += 1

        # Close deadline
        if days_to_deadline[i] <= 3:
            score += 3
        elif days_to_deadline[i] <= 7:
            score += 1.5
        elif days_to_deadline[i] <= 14:
            score += 0.5

        # High workload
        if workload[i] >= 5:
            score += 1.5
        elif workload[i] >= 4:
            score += 0.5

        # Add noise
        score += rng.normal(0, 0.8)

        if score >= 6:
            risk.append("high")
        elif score >= 3.5:
            risk.append("medium")
        else:
            risk.append("low")

    df = pd.DataFrame({
        "gender": genders,
        "workload": workload,
        "study_hours": study_hours,
        "confidence": confidence,
        "stress": stress,
        "days_to_deadline": days_to_deadline,
        "risk": risk,
    })

    return df


# Generate the dataset
df = generate_dataset()
print(f"Generated {len(df)} student records")
print(f"\nRisk distribution:\n{df['risk'].value_counts()}")
df.head(10)

### 1.1 Dataset Distribution Visualisation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Feature Distributions by Risk Level", fontsize=16, fontweight="bold")

features = ["stress", "confidence", "study_hours", "workload", "days_to_deadline"]
colors = {"low": "#2ecc71", "medium": "#f39c12", "high": "#e74c3c"}

for idx, feat in enumerate(features):
    ax = axes[idx // 3][idx % 3]
    for risk_level in ["low", "medium", "high"]:
        subset = df[df["risk"] == risk_level]
        ax.hist(subset[feat], bins=20, alpha=0.5, label=risk_level, color=colors[risk_level])
    ax.set_title(feat.replace("_", " ").title(), fontsize=12)
    ax.legend()

# Risk distribution pie chart
ax = axes[1][2]
risk_counts = df["risk"].value_counts()
ax.pie(risk_counts.values, labels=risk_counts.index, autopct="%1.1f%%",
       colors=[colors[r] for r in risk_counts.index], startangle=90)
ax.set_title("Risk Distribution", fontsize=12)

plt.tight_layout()
plt.show()

---
## 2. Machine Learning Model

A **Random Forest Classifier** trained on the synthetic dataset to predict student risk level. We evaluate with accuracy, precision, recall, F1 score, confusion matrix, and feature importance.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)

FEATURE_COLS = ["gender_encoded", "workload", "study_hours", "confidence", "stress", "days_to_deadline"]


def load_and_prepare_data(df):
    le = LabelEncoder()
    df["gender_encoded"] = le.fit_transform(df["gender"])
    X = df[FEATURE_COLS]
    y = df["risk"]
    return X, y, le


def train_model(X, y, seed=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    clf = RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=seed, class_weight="balanced"
    )
    clf.fit(X_train, y_train)
    return clf, X_train, X_test, y_train, y_test


def evaluate_model(clf, X_test, y_test):
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted")
    cm = confusion_matrix(y_test, y_pred, labels=["low", "medium", "high"])
    report = classification_report(y_test, y_pred, labels=["low", "medium", "high"])
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1,
            "confusion_matrix": cm, "report": report}


def predict_risk(clf, le, gender, workload, study_hours, confidence, stress, days_to_deadline):
    gender_enc = le.transform([gender])[0]
    features = pd.DataFrame(
        [[gender_enc, workload, study_hours, confidence, stress, days_to_deadline]],
        columns=FEATURE_COLS,
    )
    prediction = clf.predict(features)[0]
    probabilities = clf.predict_proba(features)[0]
    prob_dict = {label: round(float(prob), 3) for label, prob in zip(clf.classes_, probabilities)}
    return prediction, prob_dict


def get_feature_importance(clf):
    names = ["gender", "workload", "study_hours", "confidence", "stress", "days_to_deadline"]
    return sorted(zip(names, clf.feature_importances_), key=lambda x: x[1], reverse=True)


# Train and evaluate
X, y, le = load_and_prepare_data(df)
clf, X_train, X_test, y_train, y_test = train_model(X, y)
metrics = evaluate_model(clf, X_test, y_test)

print("=== Model Evaluation ===")
print(f"Accuracy:  {metrics['accuracy']:.3f}")
print(f"Precision: {metrics['precision']:.3f}")
print(f"Recall:    {metrics['recall']:.3f}")
print(f"F1 Score:  {metrics['f1']:.3f}")
print(f"\nClassification Report:\n{metrics['report']}")

### 2.1 Confusion Matrix Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(metrics["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
            xticklabels=["low", "medium", "high"],
            yticklabels=["low", "medium", "high"], ax=ax)
ax.set_title("Confusion Matrix — Random Forest Risk Classifier", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.tight_layout()
plt.show()

### 2.2 Feature Importance

In [ ]:
importances = get_feature_importance(clf)
names, values = zip(*importances)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names[::-1], values[::-1], color="#3498db", edgecolor="#2c3e50")
ax.set_xlabel("Importance", fontsize=12)
ax.set_title("Feature Importance — Random Forest", fontsize=14, fontweight="bold")
for bar, val in zip(bars, values[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

### 2.3 Risk Probability Bar Chart (Example Prediction)

In [ ]:
# Example prediction
pred, probs = predict_risk(clf, le, "female", 4, 10, 4.0, 7.5, 5)
print(f"Prediction: {pred}")
print(f"Probabilities: {probs}")

fig, ax = plt.subplots(figsize=(8, 4))
risk_colors = {"low": "#2ecc71", "medium": "#f39c12", "high": "#e74c3c"}
labels = list(probs.keys())
vals = list(probs.values())
bar_colors = [risk_colors.get(l, "#95a5a6") for l in labels]
ax.bar(labels, vals, color=bar_colors, edgecolor="#2c3e50", width=0.5)
ax.set_ylabel("Probability", fontsize=12)
ax.set_title(f"Risk Probability Distribution (Predicted: {pred.upper()})", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1)
for i, v in enumerate(vals):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 3. Rule-Based Expert System

A **forward chaining** expert system with **22 rules** covering stress, confidence, workload, deadline urgency, and combined risk factors. **Backward chaining** provides explanations for why a particular risk level was assigned.

In [ ]:
class Rule:
    def __init__(self, rule_id, description, conditions, conclusions):
        """
        conditions: dict of fact_name -> callable(value) -> bool
        conclusions: list of (fact_name, value) to assert
        """
        self.rule_id = rule_id
        self.description = description
        self.conditions = conditions
        self.conclusions = conclusions

    def matches(self, facts):
        for fact_name, test_fn in self.conditions.items():
            if fact_name not in facts:
                return False
            if not test_fn(facts[fact_name]):
                return False
        return True

    def __repr__(self):
        return f"Rule({self.rule_id}: {self.description})"


# ── Define all 22 Rules ──────────────────────────────────────

RULES = [
    # Stress rules
    Rule("R1", "Very high stress indicates burnout risk",
         {"stress": lambda v: v >= 8},
         [("stress_level", "very_high"), ("burnout_risk", True)]),
    Rule("R2", "High stress detected",
         {"stress": lambda v: 6 <= v < 8},
         [("stress_level", "high")]),
    Rule("R3", "Moderate stress detected",
         {"stress": lambda v: 4 <= v < 6},
         [("stress_level", "moderate")]),
    Rule("R4", "Low stress detected",
         {"stress": lambda v: v < 4},
         [("stress_level", "low")]),

    # Confidence rules
    Rule("R5", "Very low confidence \u2014 needs encouragement",
         {"confidence": lambda v: v <= 3},
         [("confidence_level", "very_low"), ("needs_encouragement", True)]),
    Rule("R6", "Low confidence detected",
         {"confidence": lambda v: 3 < v <= 5},
         [("confidence_level", "low")]),
    Rule("R7", "Good confidence detected",
         {"confidence": lambda v: 5 < v <= 7.5},
         [("confidence_level", "good")]),
    Rule("R8", "High confidence detected",
         {"confidence": lambda v: v > 7.5},
         [("confidence_level", "high")]),

    # Deadline rules
    Rule("R9", "Urgent deadline \u2014 within 3 days",
         {"days_to_deadline": lambda v: v <= 3},
         [("deadline_urgency", "critical"), ("needs_immediate_action", True)]),
    Rule("R10", "Close deadline \u2014 within 7 days",
         {"days_to_deadline": lambda v: 3 < v <= 7},
         [("deadline_urgency", "high")]),
    Rule("R11", "Approaching deadline \u2014 within 14 days",
         {"days_to_deadline": lambda v: 7 < v <= 14},
         [("deadline_urgency", "moderate")]),
    Rule("R12", "Deadline is far away",
         {"days_to_deadline": lambda v: v > 14},
         [("deadline_urgency", "low")]),

    # Workload rules
    Rule("R13", "Heavy workload \u2014 5+ modules",
         {"workload": lambda v: v >= 5},
         [("workload_level", "heavy"), ("overloaded", True)]),
    Rule("R14", "Moderate workload \u2014 3-4 modules",
         {"workload": lambda v: 3 <= v < 5},
         [("workload_level", "moderate")]),
    Rule("R15", "Light workload \u2014 1-2 modules",
         {"workload": lambda v: v < 3},
         [("workload_level", "light")]),

    # Study hours relative to workload
    Rule("R16", "Insufficient study time for workload",
         {"study_hours": lambda v: v is not None, "workload": lambda v: v is not None},
         []),  # conclusions added dynamically

    # Combined risk rules
    Rule("R17", "High risk: high stress + low confidence",
         {"stress_level": lambda v: v in ("very_high", "high"),
          "confidence_level": lambda v: v in ("very_low", "low")},
         [("risk_flag", "high"), ("recommendation", "Seek academic support and reduce workload if possible")]),

    Rule("R18", "High risk: critical deadline + heavy workload",
         {"deadline_urgency": lambda v: v == "critical",
          "workload_level": lambda v: v == "heavy"},
         [("risk_flag", "high"), ("recommendation", "Prioritise the nearest deadline and defer non-urgent tasks")]),

    Rule("R19", "Medium risk: high stress or low confidence alone",
         {"stress_level": lambda v: v in ("very_high", "high")},
         [("risk_contributing_factor", "stress")]),

    Rule("R20", "Low risk: good balance across factors",
         {"stress_level": lambda v: v in ("low", "moderate"),
          "confidence_level": lambda v: v in ("good", "high"),
          "deadline_urgency": lambda v: v in ("low", "moderate")},
         [("risk_flag", "low"), ("recommendation", "You are on track \u2014 maintain your current routine")]),

    Rule("R21", "Burnout risk requires rest recommendation",
         {"burnout_risk": lambda v: v is True},
         [("recommendation", "Schedule regular breaks to avoid burnout")]),

    Rule("R22", "Encouragement needed for low confidence",
         {"needs_encouragement": lambda v: v is True},
         [("recommendation", "Start with easier tasks to build momentum and confidence")]),
]


def _check_study_ratio(facts):
    """Special logic for R16 \u2014 study hours vs workload ratio."""
    if "study_hours" in facts and "workload" in facts:
        ratio = facts["study_hours"] / max(facts["workload"], 1)
        if ratio < 3:
            return [("study_adequacy", "insufficient"),
                    ("recommendation", "Increase weekly study hours \u2014 aim for at least 3 hours per module")]
        elif ratio < 5:
            return [("study_adequacy", "borderline")]
        else:
            return [("study_adequacy", "adequate")]
    return []


# ── Forward Chaining ──────────────────────────────────────────

def forward_chain(facts):
    """
    Iteratively fire rules until no new facts are derived.
    Returns: (derived_facts, fired_rules)
    """
    working = dict(facts)
    fired = []
    fired_ids = set()
    changed = True

    while changed:
        changed = False
        for rule in RULES:
            if rule.rule_id in fired_ids:
                continue

            if rule.rule_id == "R16":
                extra = _check_study_ratio(working)
                if extra:
                    new_facts = []
                    for fname, fval in extra:
                        if fname not in working or (fname == "recommendation" and fval != working.get(fname)):
                            working.setdefault(fname, fval)
                            new_facts.append((fname, fval))
                    if new_facts:
                        fired.append((rule, new_facts))
                        fired_ids.add(rule.rule_id)
                        changed = True
                continue

            if rule.matches(working):
                new_facts = []
                for fname, fval in rule.conclusions:
                    if fname not in working:
                        working[fname] = fval
                        new_facts.append((fname, fval))
                if new_facts:
                    fired.append((rule, new_facts))
                    fired_ids.add(rule.rule_id)
                    changed = True

    return working, fired


# ── Backward Chaining ─────────────────────────────────────────

def backward_chain(facts, goal_fact, goal_value=None):
    """
    Given a goal (e.g., risk_flag=high), trace back which rules and base facts
    would lead to that conclusion.
    """
    explanations = []
    _backward_recurse(facts, goal_fact, goal_value, explanations, visited=set())
    return explanations


def _backward_recurse(facts, goal_fact, goal_value, explanations, visited):
    for rule in RULES:
        if rule.rule_id in visited:
            continue

        concludes_goal = False
        for fname, fval in rule.conclusions:
            if fname == goal_fact:
                if goal_value is None or fval == goal_value:
                    concludes_goal = True
                    break

        if not concludes_goal:
            continue

        visited.add(rule.rule_id)

        if rule.matches(facts):
            condition_strs = []
            for fact_name in rule.conditions:
                if fact_name in facts:
                    condition_strs.append(f"{fact_name} = {facts[fact_name]}")
            explanations.append(
                f"[{rule.rule_id}] {rule.description} "
                f"(because {', '.join(condition_strs)})"
            )
            for fact_name in rule.conditions:
                _backward_recurse(facts, fact_name, facts.get(fact_name), explanations, visited)


def assess_student(student_data):
    """
    Main entry point: take raw student data, run forward chaining,
    collect risk level and recommendations.
    """
    facts = dict(student_data)
    all_facts, fired = forward_chain(facts)

    risk = all_facts.get("risk_flag", "medium")

    recommendations = []
    for rule, new_facts in fired:
        for fname, fval in new_facts:
            if fname == "recommendation":
                recommendations.append(fval)

    if not recommendations:
        if risk == "medium":
            recommendations.append("Consider balancing your workload and managing stress proactively.")

    return {
        "risk_level": risk,
        "recommendations": recommendations,
        "fired_rules": [(r.rule_id, r.description) for r, _ in fired],
        "all_facts": all_facts,
    }


def explain_risk(all_facts, risk_level):
    """Use backward chaining to explain why a risk level was assigned."""
    explanations = backward_chain(all_facts, "risk_flag", risk_level)
    if not explanations:
        for goal in ["stress_level", "confidence_level", "deadline_urgency", "workload_level", "study_adequacy"]:
            if goal in all_facts:
                more = backward_chain(all_facts, goal, all_facts[goal])
                explanations.extend(more)
    return explanations


print("Rule engine loaded with", len(RULES), "rules")

### 3.1 Rule Engine Demo

In [ ]:
# Demo: High-risk student
student = {
    "stress": 8.5,
    "confidence": 3.0,
    "days_to_deadline": 4,
    "workload": 5,
    "study_hours": 8,
    "gender": "female",
}

print("=" * 60)
print("FORWARD CHAINING")
print("=" * 60)
result = assess_student(student)
print(f"Risk Level: {result['risk_level'].upper()}")
print(f"\nRecommendations:")
for r in result["recommendations"]:
    print(f"  \u2022 {r}")
print(f"\nFired Rules ({len(result['fired_rules'])}):\n")
for rid, desc in result["fired_rules"]:
    print(f"  [{rid}] {desc}")

print(f"\n{'=' * 60}")
print("BACKWARD CHAINING (Explanation)")
print("=" * 60)
explanations = explain_risk(result["all_facts"], result["risk_level"])
for exp in explanations:
    print(f"  {exp}")

---
## 4. Search-Based Study Planner

Compares **A* Search** vs **Greedy Best-First Search** for generating optimal weekly study schedules. The heuristic combines deadline urgency and subject difficulty.

In [ ]:
import heapq
import copy
from dataclasses import dataclass, field


DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]


@dataclass
class Subject:
    name: str
    hours_needed: float
    days_to_deadline: int
    difficulty: float  # 1-10 scale

    @property
    def urgency_score(self):
        """Higher = more urgent. Combines deadline proximity and difficulty."""
        deadline_factor = max(0, 10 - self.days_to_deadline) / 10
        difficulty_factor = self.difficulty / 10
        return 0.6 * deadline_factor + 0.4 * difficulty_factor


@dataclass(order=True)
class ScheduleState:
    priority: float
    schedule: dict = field(compare=False)
    remaining: dict = field(compare=False)
    cost: float = field(default=0.0, compare=False)
    path: list = field(default_factory=list, compare=False)

    def is_goal(self):
        return all(h <= 0.01 for h in self.remaining.values())

    def get_schedule_copy(self):
        return {day: list(slots) for day, slots in self.schedule.items()}


def heuristic(state, subjects_map):
    """Estimates remaining cost to reach goal."""
    h = 0.0
    for subj_name, hours_left in state.remaining.items():
        if hours_left > 0:
            subj = subjects_map[subj_name]
            h += hours_left * (1 + subj.urgency_score)
    return h


def get_successors(state, subjects_map, max_hours_per_day, day_hours):
    """Generate next states by allocating one block (1 hour) of a subject to a day."""
    successors = []
    for day in DAYS:
        current_day_hours = sum(h for _, h in state.schedule[day])
        available = day_hours.get(day, max_hours_per_day) - current_day_hours
        if available < 1:
            continue

        for subj_name, hours_left in state.remaining.items():
            if hours_left <= 0.01:
                continue

            subj = subjects_map[subj_name]
            block = min(1.0, hours_left, available)

            new_schedule = state.get_schedule_copy()
            new_schedule[day].append((subj_name, block))

            new_remaining = dict(state.remaining)
            new_remaining[subj_name] = round(hours_left - block, 2)

            action = f"Allocate {block}h of {subj_name} on {day}"
            new_path = state.path + [action]

            step_cost = block * (1 - 0.1 * subj.urgency_score)

            successors.append(ScheduleState(
                priority=0,
                schedule=new_schedule,
                remaining=new_remaining,
                cost=state.cost + step_cost,
                path=new_path,
            ))
    return successors


def astar_search(subjects, max_hours_per_day=6, day_hours=None):
    """A* search: f(n) = g(n) + h(n)"""
    if day_hours is None:
        day_hours = {d: max_hours_per_day for d in DAYS}

    subjects_map = {s.name: s for s in subjects}
    initial_remaining = {s.name: s.hours_needed for s in subjects}
    initial_schedule = {day: [] for day in DAYS}

    start = ScheduleState(priority=0, schedule=initial_schedule,
                          remaining=initial_remaining, cost=0.0, path=[])
    h = heuristic(start, subjects_map)
    start.priority = 0 + h

    frontier = [start]
    explored = 0
    max_explored = 5000

    while frontier and explored < max_explored:
        current = heapq.heappop(frontier)
        explored += 1

        if current.is_goal():
            return current.schedule, {"algorithm": "A*", "nodes_explored": explored}

        for succ in get_successors(current, subjects_map, max_hours_per_day, day_hours):
            h = heuristic(succ, subjects_map)
            succ.priority = succ.cost + h
            heapq.heappush(frontier, succ)

    return None, {"algorithm": "A*", "nodes_explored": explored, "status": "limit_reached"}


def greedy_search(subjects, max_hours_per_day=6, day_hours=None):
    """Greedy Best-First: f(n) = h(n) only (ignores path cost)"""
    if day_hours is None:
        day_hours = {d: max_hours_per_day for d in DAYS}

    subjects_map = {s.name: s for s in subjects}
    initial_remaining = {s.name: s.hours_needed for s in subjects}
    initial_schedule = {day: [] for day in DAYS}

    start = ScheduleState(priority=0, schedule=initial_schedule,
                          remaining=initial_remaining, cost=0.0, path=[])
    start.priority = heuristic(start, subjects_map)

    frontier = [start]
    explored = 0
    max_explored = 5000

    while frontier and explored < max_explored:
        current = heapq.heappop(frontier)
        explored += 1

        if current.is_goal():
            return current.schedule, {"algorithm": "Greedy Best-First", "nodes_explored": explored}

        for succ in get_successors(current, subjects_map, max_hours_per_day, day_hours):
            h = heuristic(succ, subjects_map)
            succ.priority = h
            heapq.heappush(frontier, succ)

    return None, {"algorithm": "Greedy Best-First", "nodes_explored": explored, "status": "limit_reached"}


def create_schedule(subjects_data, available_hours_per_day=6, day_hours=None):
    """High-level function: runs both algorithms and returns comparison."""
    subjects = [Subject(**s) for s in subjects_data]
    astar_schedule, astar_stats = astar_search(subjects, available_hours_per_day, day_hours)
    greedy_schedule, greedy_stats = greedy_search(subjects, available_hours_per_day, day_hours)
    return {
        "astar": {"schedule": astar_schedule, "stats": astar_stats},
        "greedy": {"schedule": greedy_schedule, "stats": greedy_stats},
        "subjects": subjects_data,
    }


def format_schedule(schedule):
    """Format a schedule dict into a readable string."""
    if schedule is None:
        return "No valid schedule found within search limits."
    lines = []
    for day in DAYS:
        slots = schedule[day]
        if slots:
            merged = {}
            for subj, hours in slots:
                merged[subj] = merged.get(subj, 0) + hours
            parts = [f"{subj} ({h:.0f}h)" for subj, h in merged.items()]
            lines.append(f"  {day}: {', '.join(parts)}")
        else:
            lines.append(f"  {day}: Free")
    return "\n".join(lines)


print("Search planner loaded")

### 4.1 Search Planner Demo

In [ ]:
subjects_data = [
    {"name": "AI Coursework", "hours_needed": 6, "days_to_deadline": 5, "difficulty": 8},
    {"name": "Databases", "hours_needed": 4, "days_to_deadline": 10, "difficulty": 6},
    {"name": "Networking", "hours_needed": 3, "days_to_deadline": 14, "difficulty": 5},
]

result = create_schedule(subjects_data, available_hours_per_day=4)

print("=" * 60)
print("A* SEARCH SCHEDULE")
print("=" * 60)
print(format_schedule(result["astar"]["schedule"]))
print(f"\nNodes explored: {result['astar']['stats']['nodes_explored']}")

print(f"\n{'=' * 60}")
print("GREEDY BEST-FIRST SCHEDULE")
print("=" * 60)
print(format_schedule(result["greedy"]["schedule"]))
print(f"\nNodes explored: {result['greedy']['stats']['nodes_explored']}")

### 4.2 Algorithm Comparison Table

In [ ]:
comparison_data = {
    "Property": [
        "Evaluation Function",
        "Optimal?",
        "Complete?",
        "Time Complexity",
        "Space Complexity",
        "Nodes Explored (this run)",
        "Best For"
    ],
    "A* Search": [
        "f(n) = g(n) + h(n)",
        "Yes (with admissible heuristic)",
        "Yes",
        "O(b^d)",
        "O(b^d)",
        str(result["astar"]["stats"]["nodes_explored"]),
        "Finding the best schedule"
    ],
    "Greedy Best-First": [
        "f(n) = h(n)",
        "No",
        "No (may get stuck)",
        "O(b^m)",
        "O(b^m)",
        str(result["greedy"]["stats"]["nodes_explored"]),
        "Finding a quick schedule"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df.set_index("Property", inplace=True)
comparison_df

---
## 5. Fuzzy Logic Risk Assessment

An optional **fuzzy inference system** using scikit-fuzzy that provides more nuanced risk categorisation with soft boundaries between membership classes, instead of hard thresholds.

In [ ]:
import numpy as np

try:
    import skfuzzy as fuzz
    from skfuzzy import control as ctrl
    FUZZY_AVAILABLE = True
except ImportError:
    FUZZY_AVAILABLE = False
    print("scikit-fuzzy not available. Install with: pip install scikit-fuzzy")


def create_fuzzy_system():
    """Build the fuzzy inference system for risk assessment."""
    if not FUZZY_AVAILABLE:
        return None

    # Define universe of discourse
    stress = ctrl.Antecedent(np.arange(0, 11, 0.1), "stress")
    confidence = ctrl.Antecedent(np.arange(0, 11, 0.1), "confidence")
    risk = ctrl.Consequent(np.arange(0, 11, 0.1), "risk")

    # Stress membership functions
    stress["low"] = fuzz.trimf(stress.universe, [0, 0, 4])
    stress["moderate"] = fuzz.trimf(stress.universe, [3, 5, 7])
    stress["high"] = fuzz.trimf(stress.universe, [6, 8, 10])
    stress["very_high"] = fuzz.trimf(stress.universe, [8, 10, 10])

    # Confidence membership functions
    confidence["very_low"] = fuzz.trimf(confidence.universe, [0, 0, 3])
    confidence["low"] = fuzz.trimf(confidence.universe, [2, 4, 6])
    confidence["good"] = fuzz.trimf(confidence.universe, [5, 7, 9])
    confidence["high"] = fuzz.trimf(confidence.universe, [7, 10, 10])

    # Risk output membership functions
    risk["low"] = fuzz.trimf(risk.universe, [0, 0, 4])
    risk["medium"] = fuzz.trimf(risk.universe, [3, 5, 7])
    risk["high"] = fuzz.trimf(risk.universe, [6, 10, 10])

    # Fuzzy rules
    rules = [
        ctrl.Rule(stress["very_high"] & confidence["very_low"], risk["high"]),
        ctrl.Rule(stress["high"] & confidence["low"], risk["high"]),
        ctrl.Rule(stress["high"] & confidence["very_low"], risk["high"]),
        ctrl.Rule(stress["moderate"] & confidence["low"], risk["medium"]),
        ctrl.Rule(stress["moderate"] & confidence["good"], risk["medium"]),
        ctrl.Rule(stress["high"] & confidence["good"], risk["medium"]),
        ctrl.Rule(stress["low"] & confidence["good"], risk["low"]),
        ctrl.Rule(stress["low"] & confidence["high"], risk["low"]),
        ctrl.Rule(stress["moderate"] & confidence["high"], risk["low"]),
        ctrl.Rule(stress["low"] & confidence["low"], risk["medium"]),
        ctrl.Rule(stress["low"] & confidence["very_low"], risk["medium"]),
        ctrl.Rule(stress["very_high"] & confidence["high"], risk["medium"]),
        ctrl.Rule(stress["very_high"] & confidence["good"], risk["high"]),
        ctrl.Rule(stress["moderate"] & confidence["very_low"], risk["high"]),
        ctrl.Rule(stress["high"] & confidence["high"], risk["medium"]),
        ctrl.Rule(stress["very_high"] & confidence["low"], risk["high"]),
    ]

    system = ctrl.ControlSystem(rules)
    simulator = ctrl.ControlSystemSimulation(system)
    return simulator


def fuzzy_risk_assessment(stress_val, confidence_val, simulator=None):
    """
    Run fuzzy inference to get a risk score (0-10) and category.
    Returns dict with risk_score, risk_category, membership_info or None.
    """
    if not FUZZY_AVAILABLE or simulator is None:
        return None

    stress_val = max(0.1, min(9.9, stress_val))
    confidence_val = max(0.1, min(9.9, confidence_val))

    simulator.input["stress"] = stress_val
    simulator.input["confidence"] = confidence_val

    try:
        simulator.compute()
        risk_score = simulator.output["risk"]
    except Exception:
        return None

    if risk_score >= 6.5:
        category = "high"
    elif risk_score >= 3.5:
        category = "medium"
    else:
        category = "low"

    return {
        "stress_input": stress_val,
        "confidence_input": confidence_val,
        "risk_score": round(risk_score, 2),
        "risk_category": category,
    }


print("Fuzzy logic module loaded. Available:", FUZZY_AVAILABLE)

### 5.1 Fuzzy Logic Demo

In [ ]:
if FUZZY_AVAILABLE:
    sim = create_fuzzy_system()

    test_cases = [
        (9, 2, "Very high stress, very low confidence"),
        (5, 5, "Moderate stress, moderate confidence"),
        (2, 8, "Low stress, high confidence"),
        (7, 4, "High stress, low confidence"),
    ]

    print("=" * 60)
    print("FUZZY LOGIC RISK ASSESSMENT")
    print("=" * 60)
    for stress_val, conf_val, desc in test_cases:
        result = fuzzy_risk_assessment(stress_val, conf_val, sim)
        if result:
            print(f"\n  {desc}")
            print(f"    Stress={stress_val}, Confidence={conf_val}")
            print(f"    -> Risk Score: {result['risk_score']:.2f} ({result['risk_category'].upper()})")
        else:
            print(f"  {desc}: Could not compute")
else:
    print("Fuzzy logic not available. Skipping demo.")
    sim = None

---
## 6. Complete Demo: All Components Combined

Enter a student profile below and see results from **all five AI components** in one place.

> **To try your own values:** Edit the variables in the cell below and re-run it.

In [ ]:
# ================================================================
# MODIFY THESE VALUES to test different student profiles
# ================================================================
STUDENT_GENDER = "female"          # "male", "female", or "non-binary"
STUDENT_WORKLOAD = 5               # Number of active modules (1-6)
STUDENT_STUDY_HOURS = 8.0          # Weekly study hours (2-40)
STUDENT_CONFIDENCE = 3.0           # Self-reported confidence (1-10)
STUDENT_STRESS = 8.5               # Self-reported stress (1-10)
STUDENT_DAYS_TO_DEADLINE = 4       # Days until nearest deadline (1-60)
# ================================================================

print("=" * 70)
print("         STUDENT SUCCESS COPILOT — COMPREHENSIVE ASSESSMENT")
print("=" * 70)
print(f"\n  Student Profile:")
print(f"    Gender:           {STUDENT_GENDER}")
print(f"    Workload:         {STUDENT_WORKLOAD} modules")
print(f"    Study Hours/Week: {STUDENT_STUDY_HOURS}")
print(f"    Confidence:       {STUDENT_CONFIDENCE}/10")
print(f"    Stress:           {STUDENT_STRESS}/10")
print(f"    Days to Deadline: {STUDENT_DAYS_TO_DEADLINE}")

# ── 1. ML Prediction ──
print(f"\n{'─' * 70}")
print("  [1] ML MODEL PREDICTION (Random Forest)")
print(f"{'─' * 70}")
ml_pred, ml_probs = predict_risk(
    clf, le, STUDENT_GENDER, STUDENT_WORKLOAD, STUDENT_STUDY_HOURS,
    STUDENT_CONFIDENCE, STUDENT_STRESS, STUDENT_DAYS_TO_DEADLINE
)
print(f"    Predicted Risk: {ml_pred.upper()}")
print(f"    Probabilities:  {ml_probs}")

# ── 2. Rule Engine ──
print(f"\n{'─' * 70}")
print("  [2] RULE-BASED EXPERT SYSTEM (22 rules)")
print(f"{'─' * 70}")
student_data = {
    "stress": STUDENT_STRESS,
    "confidence": STUDENT_CONFIDENCE,
    "days_to_deadline": STUDENT_DAYS_TO_DEADLINE,
    "workload": STUDENT_WORKLOAD,
    "study_hours": STUDENT_STUDY_HOURS,
    "gender": STUDENT_GENDER,
}
rule_result = assess_student(student_data)
print(f"    Risk Level: {rule_result['risk_level'].upper()}")
print(f"    Rules Fired: {len(rule_result['fired_rules'])}")
print(f"    Recommendations:")
for r in rule_result["recommendations"]:
    print(f"      \u2022 {r}")
print(f"\n    Backward Chaining Explanation:")
explanations = explain_risk(rule_result["all_facts"], rule_result["risk_level"])
if explanations:
    for exp in explanations:
        print(f"      {exp}")
else:
    print("      (No direct explanation chain found — risk inferred from multiple factors)")

# ── 3. Fuzzy Logic ──
print(f"\n{'─' * 70}")
print("  [3] FUZZY LOGIC ASSESSMENT")
print(f"{'─' * 70}")
if FUZZY_AVAILABLE and sim is not None:
    fuzzy_result = fuzzy_risk_assessment(STUDENT_STRESS, STUDENT_CONFIDENCE, sim)
    if fuzzy_result:
        print(f"    Risk Score: {fuzzy_result['risk_score']:.2f}/10")
        print(f"    Category:   {fuzzy_result['risk_category'].upper()}")
    else:
        print("    Could not compute fuzzy assessment")
else:
    print("    Fuzzy logic not available (install scikit-fuzzy)")

# ── 4. Study Schedule ──
print(f"\n{'─' * 70}")
print("  [4] SEARCH-BASED STUDY PLANNER")
print(f"{'─' * 70}")
subjects_for_schedule = [
    {"name": "AI Coursework", "hours_needed": 6, "days_to_deadline": 5, "difficulty": 8},
    {"name": "Databases", "hours_needed": 4, "days_to_deadline": 10, "difficulty": 6},
    {"name": "Networking", "hours_needed": 3, "days_to_deadline": 14, "difficulty": 5},
]
schedule_result = create_schedule(subjects_for_schedule, available_hours_per_day=4)
print(f"\n    A* Schedule (nodes explored: {schedule_result['astar']['stats']['nodes_explored']}):")
for line in format_schedule(schedule_result["astar"]["schedule"]).split("\n"):
    print(f"    {line}")
print(f"\n    Greedy Schedule (nodes explored: {schedule_result['greedy']['stats']['nodes_explored']}):")
for line in format_schedule(schedule_result["greedy"]["schedule"]).split("\n"):
    print(f"    {line}")

# ── Summary ──
print(f"\n{'=' * 70}")
print("  SUMMARY")
print(f"{'=' * 70}")
print(f"    ML Model says:     {ml_pred.upper()}")
print(f"    Rule Engine says:  {rule_result['risk_level'].upper()}")
if FUZZY_AVAILABLE and sim is not None and fuzzy_result:
    print(f"    Fuzzy Logic says:  {fuzzy_result['risk_category'].upper()} (score: {fuzzy_result['risk_score']:.2f})")
print(f"\n{'=' * 70}")

### 6.1 Visualisation: Risk Probability for This Student

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ML probability bar chart
ax = axes[0]
risk_colors = {"low": "#2ecc71", "medium": "#f39c12", "high": "#e74c3c"}
labels = list(ml_probs.keys())
vals = list(ml_probs.values())
bar_colors = [risk_colors.get(l, "#95a5a6") for l in labels]
ax.bar(labels, vals, color=bar_colors, edgecolor="#2c3e50", width=0.5)
ax.set_ylabel("Probability")
ax.set_title(f"ML Risk Probabilities (Predicted: {ml_pred.upper()})", fontweight="bold")
ax.set_ylim(0, 1)
for i, v in enumerate(vals):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")

# Fuzzy risk gauge
ax = axes[1]
if FUZZY_AVAILABLE and sim is not None and fuzzy_result:
    score = fuzzy_result["risk_score"]
    bar_color = "#2ecc71" if score < 3.5 else ("#f39c12" if score < 6.5 else "#e74c3c")
    ax.barh(["Fuzzy Risk"], [score], color=bar_color, edgecolor="#2c3e50", height=0.4)
    ax.set_xlim(0, 10)
    ax.set_title(f"Fuzzy Risk Score: {score:.2f}/10 ({fuzzy_result['risk_category'].upper()})", fontweight="bold")
    ax.axvline(x=3.5, color="#f39c12", linestyle="--", alpha=0.7, label="Medium threshold")
    ax.axvline(x=6.5, color="#e74c3c", linestyle="--", alpha=0.7, label="High threshold")
    ax.legend()
else:
    ax.text(0.5, 0.5, "Fuzzy logic not available", ha="center", va="center",
            transform=ax.transAxes, fontsize=14)
    ax.set_title("Fuzzy Risk Score", fontweight="bold")

plt.tight_layout()
plt.show()

---
## 7. Launch Interactive Streamlit App

Run the cell below to launch the full interactive web app with all AI components.